# EasyRAG PDF And Multimodal Loading

## Chapter position

This chapter sits before indexing. Raw files or in-memory text become canonical `Document` objects here, so mistakes at this layer tend to echo through every later stage.

## Learning question

How do PDF pages, image-only pages, and multimodal hints still end up on the same document contract as plain text files?

## Success criteria

- simulate repository PDF loading without relying on a real parser
- inspect per-page metadata, image extraction, and scanned-page placeholders
- trace how PDF-derived documents become multimodal embedding inputs

## Source code anchors

- `easyrag.rag.indexing.loaders.load_repo_documents`
- `easyrag.rag.indexing.loaders.load_pdf_documents`
- `easyrag.rag.indexing.loaders.build_document_metadata`
- `easyrag.rag.indexing.pipeline._build_embedding_input`


## Method principles

- `easyrag.rag.indexing.loaders.load_repo_documents`: This is the canonical repo discovery entry point. It filters indexable files, normalizes their text, and returns canonical `Document` objects with stable metadata.
- `easyrag.rag.indexing.loaders.load_pdf_documents`: This loader turns a PDF into page-level `Document` objects. It preserves page numbers, image paths, and visual-only placeholders so the rest of the pipeline still sees a normal document contract.
- `easyrag.rag.indexing.loaders.build_document_metadata`: This helper creates the canonical metadata shape for loaded documents. It is where path, relative path, title, `doc_id`, page number, and image hints become stable identifiers.
- `easyrag.rag.indexing.pipeline._build_embedding_input`: This helper builds the object that actually gets embedded. It is where plain text can stay plain text, while multimodal chunks can keep image paths attached for vision-language models.

Design reason: these anchors are chosen at the raw-input to canonical-document boundary. The notebook keeps that contract explicit because later chunking, embedding, and retrieval quality are only as trustworthy as the documents and metadata produced here.


## How the code fits together

The flow in this notebook is mocked PDF pages -> document metadata -> embedding inputs -> provider handoff. The goal is not to show every internal detail at once. The goal is to keep the boundary for this stage visible enough that later behavior still feels explainable. If a result looks odd, the intermediate objects in this notebook should tell you where to look next.

Design reason: the notebook establishes raw material first, then turns it into canonical documents, then inspects what survived that normalization boundary. That order makes it clear which later behaviors come from source quality and which come from later indexing or retrieval policy.


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.rag.indexing.loaders import (
    build_document_metadata,
    load_repo_documents,
    load_pdf_documents,
)
from easyrag.rag.indexing.pipeline import build_insert_payloads

## What this cell is proving

We use mocked PDF pages so the notebook stays stable on any machine. That lets us inspect the loader behavior directly instead of depending on a particular PDF parser installation.

Design reason: this cell performs the real indexing-side stage transition. Calling the helper directly keeps visible where raw inputs become canonical documents, chunks, vectors, or workspace records, which is exactly the boundary you need to inspect when later retrieval looks surprising.


In [ ]:
repo_tmp = tempfile.TemporaryDirectory()
repo_root = Path(repo_tmp.name) / "demo_repo"
docs_dir = repo_root / "docs"
docs_dir.mkdir(parents=True)
(docs_dir / "overview.md").write_text(
    "# Overview\nEasyRAG keeps PDF and text inputs on one document contract.\n",
    encoding="utf-8",
)
pdf_path = docs_dir / "manual.pdf"
pdf_path.write_bytes(b"%PDF-1.4 fake")

fake_image = mock.Mock(name="diagram.png", data=b"PNGDATA")
fake_pages = [
    mock.Mock(
        extract_text=mock.Mock(return_value="Page one architecture notes"), images=[]
    ),
    mock.Mock(extract_text=mock.Mock(return_value=""), images=[fake_image]),
    mock.Mock(
        extract_text=mock.Mock(return_value="Page three setup details"), images=[]
    ),
]

with mock.patch(
    "easyrag.rag.indexing.loaders.PdfReader", return_value=mock.Mock(pages=fake_pages)
):
    pdf_documents = load_pdf_documents(pdf_path, repo_root)
    repo_documents = load_repo_documents(repo_root)

print("PDF page documents")
for document in pdf_documents:
    print(document.page_content)
    _print_json(document.metadata)
    print()

print("All repo documents")
for document in repo_documents:
    _print_json(document.metadata)

## Why this output looks like this

The image-only page still becomes a `Document`. EasyRAG inserts a small scanned-page placeholder so the page is visible to later stages, and the metadata keeps the extracted image paths for multimodal embedding or rerank flows.

Design reason: these counts, payloads, or files are not incidental logging. They are the evidence that raw material has already been turned into stable retrieval artifacts before any later query tries to consume them.


In [ ]:
pdf_only_rag = EasyRAG(
    working_dir=repo_tmp.name,
    workspace="pdf-loading-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
pdf_payloads = build_insert_payloads(pdf_only_rag, pdf_documents)

print("Document payload for page 2")
second_page = next(
    item
    for item in pdf_payloads["documents"]
    if item["metadata"].get("page_number") == 2
)
_print_json(second_page)
print("\nVector chunk embedding inputs")
for item in pdf_payloads["vector_chunks"]:
    _print_json(
        {
            "id": item["id"],
            "embedding_input": item["embedding_input"],
            "image_paths": item["metadata"].get("image_paths", []),
        }
    )

## What this cell is proving

The provider-backed path should preserve the same contract even when the backend behavior is less deterministic.

If OpenAI-compatible settings are available, the next cell shows what a provider-backed workspace does with the same PDF-derived documents. The point is to confirm that the multimodal metadata survives the handoff.

In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_rag = EasyRAG(
        working_dir=repo_tmp.name, workspace="provider-pdf-loading-demo"
    )
    try:
        provider_payloads = build_insert_payloads(provider_rag, pdf_documents)
        for item in provider_payloads["vector_chunks"]:
            _print_json({"id": item["id"], "embedding_input": item["embedding_input"]})
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")

## Common mistakes / debugging cues

- If a later stage looks wrong, inspect `doc_id`, logical path, title, and normalized text here first.
- Do not confuse canonical `Document` preparation with actual index writes. They are different boundaries.
- Noisy or empty documents usually create problems long before retrieval. Loading and quality checks are where you catch them cheaply.

## Takeaway

The important change is not that PDF pages suddenly become plain text. The important change is that every page now has a stable `doc_id`, `page_number`, source type, and optional image payload. That lets the rest of the pipeline treat PDFs as first-class indexed documents instead of a special case.

In [ ]:
repo_tmp.cleanup()
print("Cleaned up the temporary PDF-loading workspace.")

## Where to go next

- Continue with [02_04_normalization_and_cleaning.ipynb](02_04_normalization_and_cleaning.ipynb) to see how extraction-heavy sources are normalized before indexing.
- Read [02-data-loading-overview.md](../../docs/02-data-loading-overview.md) for the chapter-level loading map.
- Revisit [00_02_observing_rag_artifacts.ipynb](../00_overview/00_02_observing_rag_artifacts.ipynb) if you want to compare PDF artifacts with the end-to-end lifecycle view.